# Regressione logistica

In [1]:
import pandas as pd
import numpy as np
import re
import time
import warnings
from pathlib import Path
from datetime import timedelta
from tabulate import tabulate
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score


# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
# Lista dei csv su cui fare training
# datasets = {
#     'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
#     'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
#     'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv',
#     'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
#     't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
#     't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
#     't2_original': FILE_PATH / 't2_original_masks.csv',
#     'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
#     'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
#     'original_dynamic': FILE_PATH / 'original_dynamic.csv'
# }

datasets = {
    'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
    'duke_lesions' : FILE_PATH / 'duke_lesions.csv',
    'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
    'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
}


# datasets = {
#     'duke_lesions_radiomic' : FILE_PATH / 'duke_lesions_radiomic_medsam.csv',
#     'duke_lesions' : FILE_PATH / 'duke_lesions.csv'
# }"""


# datasets = {
#     'ambl_lesions_radiomic' : FILE_PATH / 'ambl_lesions_radiomic_medsam.csv',
#     'ambl_lesions' : FILE_PATH / 'ambl_lesions.csv'
# }

# Training Duke

In [2]:
def training_duke(file_path, csv_name):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    # --- Controllo colonne target attese ---
    required_targets = ["ER", "PR", "HER2"]
    missing = [c for c in required_targets if c not in df_validi.columns]
    if missing:
        print(f"[ERRORE] {csv_name} - mancano colonne target: {missing}")
        return None

    # --- Creo i target binari direttamente (già binari nel Duke) ---
    final_target_list = ["ER_class", "PR_class", "HER2_class"]

    df_validi["ER_class"]   = pd.to_numeric(df_validi["ER"], errors="coerce")
    df_validi["PR_class"]   = pd.to_numeric(df_validi["PR"], errors="coerce")
    df_validi["HER2_class"] = pd.to_numeric(df_validi["HER2"], errors="coerce")

    # Tengo solo righe con tutti i target presenti e casto a int
    df_validi = df_validi.dropna(subset=final_target_list).copy()
    for col in final_target_list:
        df_validi[col] = df_validi[col].astype(int)

    # --- Tolgo target con 1 sola classe ---
    targets_da_rimuovere = []
    for col in final_target_list:
        if df_validi[col].nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # --- Features / Target / Groups ---
    raw_target_cols = ["ER", "PR", "HER2"]

    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign", "GRADE", "isTN", "Breast"
    ] + raw_target_cols + final_target_list

    features = df_validi.drop(columns=features_to_drop, errors="ignore")
    target = df_validi[final_target_list]
    groups = df_validi["Patient ID"]

    # Imputazione features numeriche
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.fillna(features.mean(numeric_only=True))

    # Pulizia nomi colonne
    features.columns = [re.sub(r"\[|\]|<", "", col) for col in features.columns]

    # --- StratifiedGroupKFold ---
    if "HER2_class" in final_target_list:
        y_strat = target["HER2_class"].astype(str)
        n_pos = int(target["HER2_class"].sum())
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg("_".join, axis=1)
        n_splits = 5


     # Debug
    min_class_count = y_strat.value_counts().min()
    n_splits = min(n_splits, min_class_count)

    if n_splits < 2:
        print(f"[ERRORE] {csv_name} - troppo pochi campioni per CV stratificata")
        return None

    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    if len(rare) > 0:
        y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(sgkf.split(features, y_strat, groups=groups))

    # Debug utilissimo: controlla positivi HER2 per fold
    """if "HER2_class" in final_target_list:
        for k, (tr, te) in enumerate(splits):
            tr_pos = int(target.iloc[tr]["HER2_class"].sum())
            te_pos = int(target.iloc[te]["HER2_class"].sum())
            print(f"[{csv_name}] Fold {k}: HER2 train pos={tr_pos} | test pos={te_pos}")"""

    # Pipeline: StandardScaler + LogisticRegression (L2 fissa)
    logistic_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            random_state=42,
            n_jobs=1,
            class_weight=None,
            solver='liblinear',
            penalty='l2',
            max_iter=2000,
            tol=1e-4
        ))
    ])

    multi_output_model = MultiOutputClassifier(logistic_pipeline)

    iperparametri = {
        'estimator__classifier__C': [0.001, 0.01, 0.1]
    }

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return float(np.mean(scores))

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = len(iperparametri['estimator__classifier__C'])
    #print(f"\nInizio Grid Search Logistic (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'overflow encountered')
        warnings.filterwarnings('ignore', 'invalid value encountered')
        warnings.filterwarnings('ignore', 'ConvergenceWarning')

        grid_search = GridSearchCV(
            estimator=multi_output_model,
            param_grid=iperparametri,
            cv=splits,
            scoring=scorer,
            n_jobs=-1,
            verbose=1,
            refit=True,
            return_train_score=False,
            error_score='raise'
        )

        grid_search.fit(features, target)

    # --- RECUPERO RISULTATI ---
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    # Pulisco i nomi dei parametri (solo C) e aggiungo penalty='l2' fisso
    clean_param_dict = {
        'C': best_params['estimator__classifier__C'],
        'penalty': 'l2'
    }

    # Metriche per FOLD e per LABEL
    fold_reports = []

    for k, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Pipeline fresca per ogni fold
        fresh_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                random_state=42,
                n_jobs=1,
                class_weight=None,
                solver='liblinear',
                max_iter=2000,
                tol=1e-4,
                penalty='l2',
                C=clean_param_dict['C']
            ))
        ])

        model_clone = MultiOutputClassifier(fresh_pipeline)
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)



        # DEBUG: Fold k
        print(f"\n[DEBUG] Fold {k}")
        for i, col in enumerate(final_target_list):
            proba_i = y_proba_list[i]
            y_true_i = y_test.iloc[:, i].values

            print(
                f"  Target: {col} | "
                f"y_true classes: {np.unique(y_true_i)} | "
                f"proba shape: {proba_i.shape}"
            )



        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i]
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            auc_val = np.nan
            if len(np.unique(y_true_i)) == 2:
                proba_i = y_proba_list[i]
                if proba_i.shape[1] == 2:
                    auc_val = roc_auc_score(y_true_i, proba_i[:, 1])

            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'auc': auc_val
            }
        
        # Debug
        print(f"\nMetriche Fold {k}")
        for col, m in fold_metrics.items():
            auc_str = "nan" if np.isnan(m["auc"]) else f"{m['auc']:.3f}"
            print(
                f"  {col}: "
                f"F1={m['f1']:.3f} | "
                f"ACC={m['accuracy']:.3f} | "
                f"AUC={auc_str}"
            )

        fold_reports.append(fold_metrics)

    final_result = {
        # Best hyperparam trovato dal grid
        "C": clean_param_dict["C"],

        # Parametri fissi (per chiarezza in tabella tesi)
        "penalty": "l2",
        "solver": "liblinear",
        "max_iter": 2000,
        "tol": 1e-4,
        "class_weight": None,
        "random_state": 42,

        # Info su CV e dataset (evita NaN->int)
        "n_splits_used": int(n_splits),
        "n_rows_used": int(len(df_validi)),

        # Risultati grid + report fold
        "mean_score": float(best_score),
        "std_score": float(grid_search.cv_results_["std_test_score"][grid_search.best_index_]),
        "fold_reports": fold_reports,
        "targets_used": final_target_list,


        # Debug
        "cv_results": grid_search.cv_results_
    }


    return final_result

# Training ambl

In [3]:
def training_ambl(file_path, csv_name):
    df = pd.read_csv(file_path)

    if "Patient ID" not in df.columns:
        raise ValueError(f"{csv_name} - manca 'Patient ID' necessario per Group split")

    df_validi = df.copy()

    # --- Controllo colonne target attese ---
    required_targets = ["ER [SII]", "PR [SII]", "HER2 [SII]"]
    missing = [c for c in required_targets if c not in df_validi.columns]
    if missing:
        print(f"[ERRORE] {csv_name} - mancano colonne target: {missing}")
        return None

    # --- Creo i target binari direttamente (già binari nel Duke) ---
    final_target_list = ["ER_class", "PR_class", "HER2_class"]

    df_validi["ER_class"]   = pd.to_numeric(df_validi["ER [SII]"], errors="coerce")
    df_validi["PR_class"]   = pd.to_numeric(df_validi["PR [SII]"], errors="coerce")
    df_validi["HER2_class"] = pd.to_numeric(df_validi["HER2 [SII]"], errors="coerce")

    # Tengo solo righe con tutti i target presenti e casto a int
    df_validi = df_validi.dropna(subset=final_target_list).copy()
    for col in final_target_list:
        df_validi[col] = df_validi[col].astype(int)

    # --- Tolgo target con 1 sola classe ---
    targets_da_rimuovere = []
    for col in final_target_list:
        if df_validi[col].nunique() < 2:
            print(f"[ATTENZIONE] {csv_name} - Target {col} ha una sola classe. Lo escludo.")
            targets_da_rimuovere.append(col)

    for col in targets_da_rimuovere:
        final_target_list.remove(col)

    if len(final_target_list) == 0:
        print(f"[ERRORE] {csv_name} - Nessun target valido (>=2 classi).")
        return None

    # --- Features / Target / Groups ---
    raw_target_cols = ["ER", "PR", "HER2"]

    features_to_drop = [
        "Patient ID", "lesion idx", "tumor/benign", "GRADE", "isTN", "Breast"
    ] + raw_target_cols + final_target_list

    features = df_validi.drop(columns=features_to_drop, errors="ignore")
    target = df_validi[final_target_list]
    groups = df_validi["Patient ID"]

    # Imputazione features numeriche
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.fillna(features.mean(numeric_only=True))

    # Pulizia nomi colonne
    features.columns = [re.sub(r"\[|\]|<", "", col) for col in features.columns]

    # --- StratifiedGroupKFold ---
    if "HER2_class" in final_target_list:
        y_strat = target["HER2_class"].astype(str)
        n_pos = int(target["HER2_class"].sum())
        n_splits = 3 if n_pos < 10 else 5
    else:
        y_strat = target.astype(int).astype(str).agg("_".join, axis=1)
        n_splits = 5


    # Debug
    min_class_count = y_strat.value_counts().min()
    n_splits = min(n_splits, min_class_count)

    if n_splits < 2:
        print(f"[ERRORE] {csv_name} - troppo pochi campioni per CV stratificata")
        return None

    vc = y_strat.value_counts()
    rare = vc[vc < n_splits].index
    if len(rare) > 0:
        y_strat = y_strat.where(~y_strat.isin(rare), other="RARE")

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    splits = list(sgkf.split(features, y_strat, groups=groups))

    # Debug utilissimo: controlla positivi HER2 per fold
    """if "HER2_class" in final_target_list:
        for k, (tr, te) in enumerate(splits):
            tr_pos = int(target.iloc[tr]["HER2_class"].sum())
            te_pos = int(target.iloc[te]["HER2_class"].sum())
            print(f"[{csv_name}] Fold {k}: HER2 train pos={tr_pos} | test pos={te_pos}")
"""
    # Pipeline: StandardScaler + LogisticRegression (L2 fissa)
    logistic_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            random_state=42,
            n_jobs=1,
            class_weight=None,
            solver='liblinear',
            penalty='l2',
            max_iter=2000,
            tol=1e-4
        ))
    ])

    multi_output_model = MultiOutputClassifier(logistic_pipeline)

    iperparametri = {
        'estimator__classifier__C': [0.001, 0.01, 0.1]
    }

    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return float(np.mean(scores))

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = len(iperparametri['estimator__classifier__C'])
    print(f"\nInizio Grid Search Logistic (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'overflow encountered')
        warnings.filterwarnings('ignore', 'invalid value encountered')
        warnings.filterwarnings('ignore', 'ConvergenceWarning')

        grid_search = GridSearchCV(
            estimator=multi_output_model,
            param_grid=iperparametri,
            cv=splits,
            scoring=scorer,
            n_jobs=-1,
            verbose=1,
            refit=True,
            return_train_score=False,
            error_score='raise'
        )

        grid_search.fit(features, target)

    # --- RECUPERO RISULTATI ---
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    # Pulisco i nomi dei parametri (solo C) e aggiungo penalty='l2' fisso
    clean_param_dict = {
        'C': best_params['estimator__classifier__C'],
        'penalty': 'l2'
    }

    # Metriche per FOLD e per LABEL
    fold_reports = []

    for k, (train_idx, test_idx) in enumerate(splits):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Pipeline fresca per ogni fold
        fresh_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                random_state=42,
                n_jobs=1,
                class_weight=None,
                solver='liblinear',
                max_iter=2000,
                tol=1e-4,
                penalty='l2',
                C=clean_param_dict['C']
            ))
        ])

        model_clone = MultiOutputClassifier(fresh_pipeline)
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)


         # DEBUG: Fold k
        print(f"\n[DEBUG] Fold {k}")
        for i, col in enumerate(final_target_list):
            proba_i = y_proba_list[i]
            y_true_i = y_test.iloc[:, i].values

            print(
                f"  Target: {col} | "
                f"y_true classes: {np.unique(y_true_i)} | "
                f"proba shape: {proba_i.shape}"
            )

        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i]
            y_pred_i = y_pred[:, i]

            f1 = f1_score(y_true_i, y_pred_i, average="macro", zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            auc_val = np.nan
            proba_i = y_proba_list[i]
            # Caso BINARIO
            if len(np.unique(y_true_i)) == 2 and proba_i.shape[1] == 2:
                auc_val = roc_auc_score(y_true_i, proba_i[:, 1])

            # Caso MULTICLASSE (AMBL)
            elif len(np.unique(y_true_i)) > 2:
                try:
                    auc_val = roc_auc_score(
                        y_true_i,
                        proba_i,
                        multi_class="ovr",
                        average="macro"
                    )
                except ValueError:
                    auc_val = np.nan


            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'auc': auc_val
            }


        # Debug
        print(f"\nMetriche Fold {k}")
        for col, m in fold_metrics.items():
            auc_str = "nan" if np.isnan(m["auc"]) else f"{m['auc']:.3f}"
            print(
                f"  {col}: "
                f"F1={m['f1']:.3f} | "
                f"ACC={m['accuracy']:.3f} | "
                f"AUC={auc_str}"
            )


        fold_reports.append(fold_metrics)

    final_result = {
        # Best hyperparam trovato dal grid
        "C": clean_param_dict["C"],

        # Parametri fissi (per chiarezza in tabella tesi)
        "penalty": "l2",
        "solver": "liblinear",
        "max_iter": 2000,
        "tol": 1e-4,
        "class_weight": None,
        "random_state": 42,

        # Info su CV e dataset (evita NaN->int)
        "n_splits_used": int(n_splits),
        "n_rows_used": int(len(df_validi)),

        # Risultati grid + report fold
        "mean_score": float(best_score),
        "std_score": float(grid_search.cv_results_["std_test_score"][grid_search.best_index_]),
        "fold_reports": fold_reports,
        "targets_used": final_target_list,


        # Debug
        "cv_results": grid_search.cv_results_ 
    }


    return final_result

# Stampo i risultati in un formato leggibile

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path


def print_grid_search_results(results_per_dataset, save_csv=True, output_path="RegressioneLogistica.csv"):
    print("\n" + "=" * 80)
    print(" " * 20 + "Metriche (MEDIA ± STD) per target")
    print("=" * 80)

    # Per il csv
    rows = []

    for dataset_name, best_result in results_per_dataset.items():
        if best_result is None:
            continue

        fold_reports = best_result["fold_reports"]
        target_names = best_result.get("targets_used", [])

        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        for target_name in target_names:
            f1_list  = np.array([fold[target_name]["f1"] for fold in fold_reports], dtype=float)
            acc_list = np.array([fold[target_name]["accuracy"] for fold in fold_reports], dtype=float)
            auc_list = np.array([fold[target_name]["auc"] for fold in fold_reports], dtype=float)

            f1_mean,  f1_std  = np.mean(f1_list),  np.std(f1_list)
            acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)

            valid_auc = ~np.isnan(auc_list)
            auc_mean = np.mean(auc_list[valid_auc]) if valid_auc.any() else np.nan
            auc_std  = np.std(auc_list[valid_auc])  if valid_auc.any() else np.nan

            # ===== STAMPA =====
            """print(f"\nTarget: {target_name}")
            print(f"  F1-score     = {f1_mean:.3f}  ±  {f1_std:.3f}")
            print(f"  Accuracy     = {acc_mean:.3f}  ±  {acc_std:.3f}")
            print(
                f"  AUC          = {auc_mean:.3f}  ±  {auc_std:.3f}"
                if not np.isnan(auc_mean)
                else f"  AUC          = NaN     ±  NaN"
            )"""

            # ===== CSV =====
            rows.append({
                "dataset": dataset_name,
                "target": target_name,
                "F1-score": f"{f1_mean:.3f} ± {f1_std:.3f}",
                "Accuracy": f"{acc_mean:.3f} ± {acc_std:.3f}",
                "AUC": (
                    f"{auc_mean:.3f} ± {auc_std:.3f}"
                    if not np.isnan(auc_mean)
                    else "NaN ± NaN"
                )
            })


    # ===== SALVATAGGIO FILE =====
    if save_csv and rows:
        df_out = pd.DataFrame(rows)
        output_path = Path(output_path)
        df_out.to_csv(output_path, index=False)
        print(f"\n Risultati salvati in: {output_path.resolve()}")

# Eseguo il tutto

In [5]:
start_time = time.time()

# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():

    name_lower = name.lower()

    if "ambl" in name_lower:
        print(f"\n>>> Training AMBL: {name}")
        results_per_dataset[name] = training_ambl(file_path, name)

    elif "duke" in name_lower:
        print(f"\n>>> Training DUKE: {name}")
        results_per_dataset[name] = training_duke(file_path, name)

    else:
        raise ValueError(f"Dataset non riconosciuto: {name}")
    
# Stampa risultati
#print_grid_search_results(results_per_dataset)

end_time = time.time()

# Tempo totale
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



>>> Training DUKE: duke_lesions_radiomic
Fitting 5 folds for each of 3 candidates, totalling 15 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


[DEBUG] Fold 0
  Target: ER_class | y_true classes: [0 1] | proba shape: (59, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (59, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (59, 2)

Metriche Fold 0
  ER_class: F1=0.609 | ACC=0.542 | AUC=0.536
  PR_class: F1=0.510 | ACC=0.576 | AUC=0.539
  HER2_class: F1=0.378 | ACC=0.610 | AUC=0.570

[DEBUG] Fold 1
  Target: ER_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (58, 2)

Metriche Fold 1
  ER_class: F1=0.639 | ACC=0.552 | AUC=0.499
  PR_class: F1=0.476 | ACC=0.431 | AUC=0.449
  HER2_class: F1=0.400 | ACC=0.586 | AUC=0.519

[DEBUG] Fold 2
  Target: ER_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (58, 2)

Metriche Fold 2
  ER_class: F1=0.600 | ACC=

/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


[DEBUG] Fold 0
  Target: ER_class | y_true classes: [0 1] | proba shape: (59, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (59, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (59, 2)

Metriche Fold 0
  ER_class: F1=0.609 | ACC=0.542 | AUC=0.539
  PR_class: F1=0.556 | ACC=0.593 | AUC=0.631
  HER2_class: F1=0.368 | ACC=0.593 | AUC=0.545

[DEBUG] Fold 1
  Target: ER_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (58, 2)

Metriche Fold 1
  ER_class: F1=0.649 | ACC=0.552 | AUC=0.506
  PR_class: F1=0.516 | ACC=0.483 | AUC=0.443
  HER2_class: F1=0.368 | ACC=0.586 | AUC=0.564

[DEBUG] Fold 2
  Target: ER_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: PR_class | y_true classes: [0 1] | proba shape: (58, 2)
  Target: HER2_class | y_true classes: [0 1] | proba shape: (58, 2)

Metriche Fold 2
  ER_class: F1=0.603 | ACC=

/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


[DEBUG] Fold 0
  Target: ER_class | y_true classes: [0 1 2 3] | proba shape: (30, 4)
  Target: PR_class | y_true classes: [0 1 2 3] | proba shape: (30, 4)
  Target: HER2_class | y_true classes: [0 1 3] | proba shape: (30, 3)

Metriche Fold 0
  ER_class: F1=0.367 | ACC=0.567 | AUC=0.614
  PR_class: F1=0.478 | ACC=0.667 | AUC=0.807
  HER2_class: F1=0.411 | ACC=0.667 | AUC=0.530

[DEBUG] Fold 1
  Target: ER_class | y_true classes: [0 1 2 3] | proba shape: (33, 4)
  Target: PR_class | y_true classes: [0 1 2 3] | proba shape: (33, 4)
  Target: HER2_class | y_true classes: [0 1 3] | proba shape: (33, 3)

Metriche Fold 1
  ER_class: F1=0.400 | ACC=0.667 | AUC=0.660
  PR_class: F1=0.530 | ACC=0.727 | AUC=0.825
  HER2_class: F1=0.621 | ACC=0.909 | AUC=0.656

>>> Training AMBL: ambl_lesions

Inizio Grid Search Logistic (GRID MINIMAL: 3 combinazioni) per: ambl_lesions
Fitting 2 folds for each of 3 candidates, totalling 6 fits

[DEBUG] Fold 0
  Target: ER_class | y_true classes: [0 1 2 3] | proba

/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-

# Validazione Multicentrica

In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

def validazione_multicentrica( df_train, df_test, feature_cols, target_col, n_splits=5, random_state=42, C=1.0):

    X_train = df_train[feature_cols].copy()
    y_train = df_train[target_col].astype(int).copy()

    X_test  = df_test[feature_cols].copy()
    y_test  = df_test[target_col].astype(int).copy()

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            random_state=random_state,
            solver='liblinear',
            max_iter=2000,
            tol=1e-4,
            penalty='l2',
            C=C
        ))
    ])

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    cv_auc = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        pipe.fit(X_tr, y_tr)
        y_val_prob = pipe.predict_proba(X_val)[:, 1]
        cv_auc.append(roc_auc_score(y_val, y_val_prob))

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    return {
        "cv_auc_mean": np.mean(cv_auc),
        "cv_auc_std":  np.std(cv_auc),
        "test_auc":    roc_auc_score(y_test, y_prob),
        "test_f1":     f1_score(y_test, y_pred),
        "test_acc":    accuracy_score(y_test, y_pred),
        "n_train":     len(df_train),
        "n_test":      len(df_test),
        "model":       pipe
    }


In [7]:
def create_targets_ambl(df):
    df = df.copy()

    df["ER_class"] = (
        pd.to_numeric(df["ER [SII]"], errors="coerce") >= 1
    ).astype(int)

    df["PR_class"] = (
        pd.to_numeric(df["PR [SII]"], errors="coerce") >= 1
    ).astype(int)

    df["HER2_class"] = (
        pd.to_numeric(df["HER2 [SII]"], errors="coerce") >= 3
    ).astype(int)

    return df


def create_targets_duke(df):

    df["ER_class"]   = pd.to_numeric(df["ER"], errors="coerce")
    df["PR_class"]   = pd.to_numeric(df["PR"], errors="coerce")
    df["HER2_class"] = pd.to_numeric(df["HER2"], errors="coerce")

    return df

In [8]:
"""
    AMBL = training (con CV interna)
    DUKE = test esterno indipendente
"""

# Carico i csv di prova
ambl = pd.read_csv("/Users/francesco/Tesi/BC-ML4/dataset/cleaned/ambl_lesions.csv")
duke = pd.read_csv("/Users/francesco/Tesi/BC-ML4/dataset/cleaned/duke_lesions.csv")

# Creo tutti i target
ambl = create_targets_ambl(ambl)
duke = create_targets_duke(duke)

# Filtro le feature che mi servono
feature_cols = sorted(
    set(ambl.columns)
    .intersection(set(duke.columns))
    - {
        "Patient ID", "lesion idx",
        "ER [SII]", "PR [SII]", "HER2 [SII]",
        "ER_class", "PR_class", "HER2_class"
    }
)
# Validazione multicentrica
targets = ["ER_class", "PR_class", "HER2_class"]
all_results = {}
for target in targets:

    ambl_t = ambl.dropna(subset=[target])
    duke_t = duke.dropna(subset=[target])

    if ambl_t[target].nunique() < 2 or duke_t[target].nunique() < 2:
        print(f"[SKIP] {target} – una sola classe")
        continue

    results = validazione_multicentrica(
        df_train=ambl_t,
        df_test=duke_t,
        feature_cols=feature_cols,
        target_col=target
    )

    all_results[target] = results


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-

In [9]:
rows = []

for target, res in all_results.items():
    rows.append({
        "Target": target.replace("_class", ""),
        "CV AUC (AMBL)": f"{res['cv_auc_mean']:.3f} ± {res['cv_auc_std']:.3f}",
        "Duke AUC": f"{res['test_auc']:.3f}",
        "Duke F1":  f"{res['test_f1']:.3f}",
        "Duke ACC": f"{res['test_acc']:.3f}",
        "N Train":  res["n_train"],
        "N Test":   res["n_test"],
    })

df = pd.DataFrame(rows)

# Ordine colonne (esplicito)
cols = [
    "Target",
    "CV AUC (AMBL)",
    "Duke AUC",
    "Duke F1",
    "Duke ACC",
    "N Train",
    "N Test",
]

df = df[cols]

# Larghezza colonne
col_widths = {
    "Target": 6,
    "CV AUC (AMBL)": 16,
    "Duke AUC": 9,
    "Duke F1": 8,
    "Duke ACC": 9,
    "N Train": 9,
    "N Test": 8,
}

def format_row(row):
    return " | ".join(
        f"{str(row[c]):<{col_widths[c]}}" for c in cols
    )

# Header
header = " | ".join(f"{c:<{col_widths[c]}}" for c in cols)
separator = "-+-".join("-" * col_widths[c] for c in cols)

print("\n=== RISULTATI DELLA VALIDAZIONE MULTICENTRICA ===\n")
print(header)
print(separator)
for _, r in df.iterrows():
    print(format_row(r))



=== RISULTATI DELLA VALIDAZIONE MULTICENTRICA ===

Target | CV AUC (AMBL)    | Duke AUC  | Duke F1  | Duke ACC  | N Train   | N Test  
-------+------------------+-----------+----------+-----------+-----------+---------
ER     | 0.525 ± 0.144    | 0.537     | 0.384    | 0.481     | 82        | 291     
PR     | 0.603 ± 0.074    | 0.505     | 0.300    | 0.502     | 82        | 291     
HER2   | 0.701 ± 0.179    | 0.544     | 0.153    | 0.656     | 82        | 291     


# Stampo la CV

In [10]:
pd.set_option("display.max_colwidth", None)

for name, res in results_per_dataset.items():
    print(f"\n### {name}")

    df = pd.DataFrame(res["cv_results"])

    display(
        df.sort_values("rank_test_score")[
            ["params", "mean_test_score", "std_test_score", "rank_test_score"]
        ].head(10)
    )



### duke_lesions_radiomic


,params,mean_test_score,std_test_score,rank_test_score
0,{'estimator__classifier__C': 0.001},0.504973,0.025576,1
2,{'estimator__classifier__C': 0.1},0.493968,0.006149,2
1,{'estimator__classifier__C': 0.01},0.488017,0.015680,3



### duke_lesions


,params,mean_test_score,std_test_score,rank_test_score
0,{'estimator__classifier__C': 0.001},0.514360,0.033643,1
1,{'estimator__classifier__C': 0.01},0.484068,0.038416,2
2,{'estimator__classifier__C': 0.1},0.462280,0.024211,3



### ambl_lesions_radiomic


,params,mean_test_score,std_test_score,rank_test_score
2,{'estimator__classifier__C': 0.1},0.467807,0.049267,1
1,{'estimator__classifier__C': 0.01},0.388972,0.002712,2
0,{'estimator__classifier__C': 0.001},0.342926,0.027190,3



### ambl_lesions


,params,mean_test_score,std_test_score,rank_test_score
2,{'estimator__classifier__C': 0.1},0.518965,0.031107,1
1,{'estimator__classifier__C': 0.01},0.411647,0.010090,2
0,{'estimator__classifier__C': 0.001},0.367400,0.036721,3
